# From Time-Series to Frequency-Domain


## Environment set up

Change the working directory to be able to work with the source-code of this repository.

In [1]:
import os
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().parents[0]
os.chdir(WORKING_DIRECTORY)

## Imports

In [7]:
from src.read import read_nasa_vibration_files_in_directory
from src.signals import processing as signal_processing
from src.signals import calculations as signal_calculations
import matplotlib.pyplot as plt
import numpy as np
from loguru import logger
import matplotlib.dates as mdates
import polars as pl

## Inputs

The inputs have been obtained from the NASA bearings documentation.

The following cell displays the data path for each test and the name of their columns:

In [ ]:
DATA_INPUTS_PER_TEST = {
    '1st_test': {'data_path': 'data/nasa_ims_bearing_dataset/1st_test',
                  'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4',
                                   'channel_5', 'channel_6', 'channel_7', 'channel_8']},
    '2nd_test': {'data_path': 'data/nasa_ims_bearing_dataset/2nd_test',
                 'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4']},
    '3rd_test': {'data_path': 'data/nasa_ims_bearing_dataset/3rd_test/4th_test/txt',
                 'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4']}
          }

As each test has a different set up of sensors or channels per bearing, the following cell describes them:

In [ ]:
BEARING_CHANNEL_MAPPING = {
    '1st_test': {'bearing_1': ['channel_1', 'channel_2'],
                 'bearing_2': ['channel_3', 'channel_4'],
                 'bearing_3': ['channel_5', 'channel_6'],
                 'bearing_4': ['channel_7', 'channel_8']},
    '2nd_test': {'bearing_1': ['channel_1'],
                 'bearing_2': ['channel_2'],
                 'bearing_3': ['channel_3'],
                 'bearing_4': ['channel_4']},
    '3rd_test': {'bearing_1': ['channel_1'],
                 'bearing_2': ['channel_2'],
                 'bearing_3': ['channel_3'],
                 'bearing_4': ['channel_4']}                 
}

Next, the faulty bearings are defined per test:

In [ ]:
FAULTY_BEARINGS_PER_TEST = {
    '1st_test': {'bearing_3': 'bearing_inner_race',
                 'bearing_4': 'bearing_roller'
                 },
    '2nd_test': {'bearing_1': 'bearing_outer_race'},
    '3rd_test': {'bearing_3': 'bearing_outer_race'}
}

As final inputs, the following parameters are needed to read properly the vibration signals. In addition, an acceptable sensor range is defined to avoid faulty channel signals:

In [ ]:
SAMPLING_FREQUENCY = 20000
MEASUREMENT_DURATION_IN_SECONDS = 1
ACCEPTABLE_SENSOR_RANGE = 0.01

## Read the data

In [ ]:
complete_data_path_per_test = {}

for test, inputs_per_test in DATA_INPUTS_PER_TEST.items():
    for key, values in inputs_per_test.items():
        data_path = inputs_per_test['data_path']
        complete_path = WORKING_DIRECTORY.joinpath(data_path)
        complete_data_path_per_test[test] = complete_path


In [ ]:
signal_resolution = signals.resolution(sampling_frequency=SAMPLING_FREQUENCY)

df_list_per_test = {}
for test, file_path in complete_data_path_per_test.items():
    logger.info(f'test: {test}')
    column_names = DATA_INPUTS_PER_TEST[test]['column_names']
    df_list = read_nasa_vibration_files_in_directory(files_path=file_path, sensors=column_names,
                                                     signal_resolution=signal_resolution,
                                                     acceptable_sensor_range=ACCEPTABLE_SENSOR_RANGE)
    df_list_per_test[test] = df_list

## Signal Processing


In [ ]:
channels_per_test = [col for col in df_list_per_test['1st_test'][0].columns if col.startswith('channel_')]
print(channels_per_test)

In [ ]:
test_df = df_list_per_test['1st_test'][0]

In [ ]:
resolution = signals.resolution(sampling_frequency=SAMPLING_FREQUENCY)
print(f'resolution: {resolution}')

In [ ]:
number_of_points =len(test_df['channel_1'].to_numpy())
print(f'number_of_points: {number_of_points}')

In [ ]:
np.linspace(start=0, num=number_of_points, stop=number_of_points * resolution)

In [ ]:
test_df

In [ ]:
test_df = df_list_per_test['2nd_test'][-1]

time = test_df['measurement_time_in_seconds'].to_numpy()
waveform = test_df['channel_1'].to_numpy()


# 1. Filter: band pass
# 2. Envelope
# 3. Power Spectrum

HIGH_PASS_FILTER_CUTOFF = 20
high_pass_filtered_signal = signals.high_pass_filter(y=waveform,
                                                     sampling_frequency=SAMPLING_RATE_IN_HERTZ,
                                                     cutoff_frequency=HIGH_PASS_FILTER_CUTOFF)

LOW_CUTOFF_FREQUENCY = 200
HIGH_CUTOFF_FREQUENCY = 4000
band_pass_filtered_signal = signals.band_pass_filter(y=waveform,
                                                     sampling_frequency=SAMPLING_FREQUENCY,
                                                     low_cutoff_frequency=LOW_CUTOFF_FREQUENCY,
                                                     high_cutoff_frequency=HIGH_CUTOFF_FREQUENCY)

enveloped_signal = signals.envelope(y=band_pass_filtered_signal)

LOW_PASS_FILTER_CUTOFF = 500
low_pass_filtered_signal = signals.low_pass_filter(y=enveloped_signal,
                                                   sampling_frequency=SAMPLING_FREQUENCY,
                                                   cutoff_frequency=LOW_PASS_FILTER_CUTOFF)

frequencies, amplitudes = signals.power_spectrum(waveform=enveloped_signal,
                                                 sampling_frequency=SAMPLING_FREQUENCY)

import pandas as pd
import plotly.express as px

# 1. Create a DataFrame for your signals
# df_signals = pd.DataFrame({
#     'Time': time,
#     'Raw': waveform,
#     'High-Pass': high_pass_filtered_signal,
#     'Band-Pass': band_pass_filtered_signal,
#     'Envelope': enveloped_signal,
#     'Low-Pass': low_pass_filtered_signal
# })

df_signals_2 = pd.DataFrame({
    'Frequency': frequencies,
    'Amplitudes': amplitudes
})

# 2. Plot all signals
# fig1 = px.line(df_signals, x='Time', y=['Raw', 'High-Pass', 'Band-Pass', 'Envelope', 'Low-Pass'],
#               title="Signal Processing Pipeline",
#               labels={'value': 'Amplitude (g)', 'variable': 'Signal Type'})

# fig1.update_layout(height=400, template='plotly_white')
# fig1.show()

fig2 = px.line(df_signals_2, x='Frequency', y='Amplitudes',
              title="Signal Processing Pipeline",
              labels={'value': 'Amplitude (g)', 'variable': 'Signal Type'})

fig2.update_layout(height=400, template='plotly_white')
fig2.show()



In [ ]:
224 * np.arange(1, 5)

In [ ]:
29.29688
166.0156
312.5


In [ ]:
test_df = df_list_per_test['1st_test'][0]

df_aggregated = test_df.group_by("file_name").agg(
    pl.col("^channel_.*$")
)

df_aggregated = df_aggregated.with_columns(
        pl.col('channel_1').map_elements(
            lambda s: signals.power_spectrum(waveform=s.to_numpy(), sampling_frequency=SAMPLING_FREQUENCY)[0],
            return_dtype=pl.List(pl.Float64)
        ).alias('power_spectrum_frequencies')
)

for channel in channels_per_test:
    df_aggregated = df_aggregated.with_columns(
        pl.col(channel).map_elements(
            lambda s: signals.power_spectrum(waveform=s.to_numpy(), sampling_frequency=SAMPLING_FREQUENCY)[1],
            return_dtype=pl.List(pl.Float64)
        ).alias(f"{channel}_power_spectrum_amplitudes")
    )

df_aggregated

In [ ]:
test_df = df_list_per_test['1st_test'][0]

df_aggregated = test_df.group_by("file_name").agg(
    pl.col("^channel_.*$")
)

df_aggregated = df_aggregated.with_columns(
        pl.col('channel_1').map_elements(
            lambda s: signals.power_spectrum(waveform=s.to_numpy(), sampling_frequency=SAMPLING_FREQUENCY)[0],
            return_dtype=pl.List(pl.Float64)
        ).alias('power_spectrum_frequencies')
)

for channel in channels_per_test:
    df_aggregated = df_aggregated.with_columns(
        pl.col(channel).map_elements(
            lambda s: signals.power_spectrum(waveform=s.to_numpy(), sampling_frequency=SAMPLING_FREQUENCY)[1],
            return_dtype=pl.List(pl.Float64)
        ).alias(f"{channel}_power_spectrum_amplitudes")
    )

df_aggregated

In [ ]:
df_aggregated = test_df.group_by("file_name").agg(
    pl.col("^channel_.*$")
)

df_final = df_aggregated.with_columns(
    pl.col("channel_1").map_elements(
        lambda s: signals.power_spectrum(waveform=s.to_numpy(), sampling_frequency=SAMPLING_FREQUENCY)[1],
        return_dtype=pl.List(pl.Float64)
    ).alias("power_spectrum_1")
)
df_final

In [ ]:
power_spectrum_df

In [ ]:
import polars as pl
import numpy as np

df = pl.DataFrame({
    "a": [1, 2, 3],
    "b": [4, 5, 6]
})

# 1. Flatten the data into a single NumPy array
# 2. Wrap it in a new DataFrame as a single row
result_df = pl.DataFrame({
    "collapsed_data": [df.to_numpy().flatten()]
})

print(result_df)

In [ ]:
pl.DataFrame(test_df.select(channels_per_test).to_numpy())

In [ ]:
pl.DataFrame({'channel_1': [test_df['channel_1'].to_numpy().flatten()]})

In [ ]:
test_df = df_list_per_test['1st_test'][0]

frequencies = signals.power_spectrum(waveform=waveform,
                                     sampling_frequency=SAMPLING_FREQUENCY)[0]

amplitudes_per_channel = []
for channel in channels_per_test:
    waveform = test_df['channel_1'].to_numpy()
    amplitudes = signals.power_spectrum(waveform=waveform,
                                        sampling_frequency=SAMPLING_FREQUENCY)[1]
    print(pl.DataFrame({f'{channel}_amplitude': amplitudes}))


In [ ]:

columns = []

waveform = df_list_per_test['1st_test'][0]['channel_1'].to_numpy()
frequencies = signals.power_spectrum(
    waveform=waveform,
    sampling_frequency=SAMPLING_FREQUENCY
)[0]

columns.append(pl.Series("frequency", frequencies))

for channel in channels_per_test:
    waveform = test_df[channel].to_numpy()
    amplitudes = signals.power_spectrum(
        waveform=waveform,
        sampling_frequency=SAMPLING_FREQUENCY
    )[1]
    
    columns.append(pl.Series(f"{channel}_amplitude", amplitudes))

final_df = pl.DataFrame(columns)

print(final_df)

In [ ]:

columns = []

frequencies = signals.power_spectrum(
    waveform=df_list_per_test['1st_test'][0]['channel_1'].to_numpy(),
    sampling_frequency=SAMPLING_FREQUENCY
)[0]

columns.append(pl.Series("frequency", frequencies))

for channel in channels_per_test:
    waveform = test_df[channel].to_numpy()
    amplitudes = signals.power_spectrum(
        waveform=waveform,
        sampling_frequency=SAMPLING_RATE_IN_HERTZ
    )[1]
    
    columns.append(pl.Series(f"{channel}_amplitude", amplitudes))

final_df = pl.DataFrame(columns)

print(final_df)

In [ ]:
power_spectrum_df